# Docling - Ingesting Documents

source:
- [docling.ai](https://docling.ai/)
- [daveebbelaar - docling](https://github.com/daveebbelaar/ai-cookbook/tree/main/knowledge/docling)

Docling is a open source python library developed by IBM.
His scope is to simplify the creation of a document ingestion pipeline:
- Parse documents (docx, pdf, html, pptx, xlsx, images, audio)
    - AI powered layout analysis
    - High performance
    - Extract metadata automatically (title, authors, ...)
- Chunking documents (creates chunks with associated metadata in a standardized form called doclang)
    - Markdown chunking
    - Hybrind chunking 
        - hierarchical 
        - + split chunks too large for the embedding moedl
        - + stitch together small chunks
        - + works with chosen tokenizers to avoid splitting tokens of you llm 

Document convertion's result is a data structure that:
- keeps together images, tables, text in a structured format
- preserves document structure and keeps the reading order
- contains chunks ready for embedding with your specific embedding model
- enhances the chunks by adding metadata like doc name, page numbers, etc...

It's also proposed as a managed SaaS that can be called via API.

## Install

```bash
    poetry add docling
```

## Usage

In [12]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer
import tiktoken
import logging  
logging.disable(logging.WARNING)

# ---------------
# Extract document content from a PDF and show it as a markdown
# ---------------

converter = DocumentConverter()
result = converter.convert("https://arxiv.org/pdf/2408.09869") # a document via a local path or URL
document = result.document

print(f"markdown output: {document.export_to_markdown()[:300]}")  # output: "## Docling Technical Report[...]"ù

# --------------
# Chunking
# --------------

MAX_TOKENS = 8191  # text-embedding-3-large's maximum context length
tokenizer = OpenAITokenizer(
    tokenizer = tiktoken.encoding_for_model("text-embedding-3-small"),
    max_tokens = MAX_TOKENS
)  # Load tokenizer for OpenAI

# Instantiate Hybrid Chunker
chunker = HybridChunker(
    tokenizer=tokenizer,
    max_tokens=MAX_TOKENS,
    merge_peers=True,
)

# Apply Hybrid Chunker
chunk_iter = chunker.chunk(dl_doc=document)
chunks = list(chunk_iter)

print(f"\nnum chunks: {len(chunks)}")
if (len(chunks) > 0):
    c = chunks[0]
    print(f"text: {c.text[:50]}")
    print(f"filename: {c.meta.origin.filename}")
    print(f"title: {c.meta.headings[0] if c.meta.headings else None}")
    print(f"page_no: {[prov.page_no for item in c.meta.doc_items for prov in item.prov]}")


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 4037.73it/s]


markdown output: <!-- image -->

## Docling Technical Report

## Version 1.0

Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta Rafael Teixeira de Lima Valery Weber Lucas 

num chunks: 21
text: Christoph Auer Maksym Lysak Ahmed Nassar Michele D
filename: 2408.09869
title: Version 1.0
page_no: [1, 1]


Note that docling is really slow and it's because it's doing OCR using pytorch with a CPU only build. 

If you've a NVIDIA GPU you can replace the dependency with the GPU (CUDA) enabled of pytorch and this would improve the performance

In [13]:
import torch
print(torch.cuda.is_available())

False
